# Practical PyTorch · II — Part 3 Companion Notebook

This goes with **"Datasets and DataLoaders: Feeding the Beast Without Choking It."** Run the cells top to bottom (Shift+Enter).

CPU is completely fine for this one — no GPU needed. Colab ships PyTorch preinstalled, so there's nothing to set up.

## 1. A tiny custom Dataset

A `Dataset` answers two questions: **how many examples** (`__len__`) and **give me example `i`** (`__getitem__`). That's the entire contract.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Some toy data: 100 examples, each with 3 features, and a label.
features = torch.randn(100, 3)
labels = torch.randint(0, 2, (100,))   # 100 labels, each 0 or 1

class MyData(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)              # how many examples

    def __getitem__(self, i):
        return self.features[i], self.labels[i]  # one (x, y) pair

ds = MyData(features, labels)
print("number of examples:", len(ds))
x0, y0 = ds[0]
print("example 0 input :", x0)
print("example 0 label :", y0)

Notice `ds[0]` hands back **one** `(x, y)` pair — not a batch, not the whole thing. Batching is the DataLoader's job.

## 2. Wrap it in a DataLoader

The `DataLoader` serves examples in **batches**, optionally **shuffled**, already stacked into tensors with a batch dimension out front.

In [ ]:
loader = DataLoader(ds, batch_size=32, shuffle=True)

# Grab the first batch to see its shape.
xb, yb = next(iter(loader))
print("batch inputs shape:", xb.shape)   # (32, 3) -- 32 examples stacked
print("batch labels shape:", yb.shape)   # (32,)
print("number of batches per epoch:", len(loader))

100 examples with `batch_size=32` gives three batches of 32 and a final batch of 4 — four batches in total. The last one is smaller, which is usually fine.

## 3. Iterating batches (the shape of a training loop)

In a real loop you iterate the loader and do one training step per batch. Here we just print the shapes so you can watch the batches roll by.

In [ ]:
for step, (xb, yb) in enumerate(loader):
    print(f"step {step}: inputs {tuple(xb.shape)}, labels {tuple(yb.shape)}")

# In a real loop, the body would be:
#     preds = model(xb)
#     loss = loss_fn(preds, yb)
#     optimizer.zero_grad(); loss.backward(); optimizer.step()

See how the last batch has 4 examples instead of 32? Pass `drop_last=True` to the DataLoader if you'd rather discard it.

## 4. Shuffle changes the order each epoch

With `shuffle=True`, every pass through the loader visits the examples in a fresh random order. Watch the first label of the first batch change across two epochs.

In [ ]:
loader = DataLoader(ds, batch_size=8, shuffle=True)

for epoch in range(2):
    first_batch_x, _ = next(iter(loader))
    print(f"epoch {epoch}: first example of first batch = {first_batch_x[0].tolist()}")

Different each time — that's shuffling doing its job. For a **test** set you'd set `shuffle=False` instead, because you want stable, repeatable evaluation.

## 5. Ready-made datasets: torchvision

Most of the time you won't write your own `Dataset`. `torchvision.datasets` gives you classic image datasets already shaped as `Dataset` objects. MNIST is tiny and downloads in seconds.

In [ ]:
from torchvision import datasets, transforms

mnist = datasets.MNIST(
    root="data", train=True, download=True,
    transform=transforms.ToTensor(),
)

print("MNIST training examples:", len(mnist))
img, label = mnist[0]
print("image tensor shape:", img.shape)   # (1, 28, 28) -- 1 channel, 28x28
print("label:", label)

# Drop it straight into a DataLoader, same as our custom one.
mnist_loader = DataLoader(mnist, batch_size=64, shuffle=True)
xb, yb = next(iter(mnist_loader))
print("batch of images:", xb.shape)       # (64, 1, 28, 28)

## 6. Ready-made datasets: Hugging Face `datasets`

For real-world text (and audio, and more), the Hugging Face `datasets` library is the usual source. One line pulls a dataset down from the Hub. This needs a quick install on Colab.

In [ ]:
!pip install -q datasets

from datasets import load_dataset

data = load_dataset("imdb")          # 50k movie reviews
print(data)
print("\nfirst training example:")
example = data["train"][0]
print("label:", example["label"])
print("text :", example["text"][:200], "...")

That's the whole data story: a `Dataset` produces one example, a `DataLoader` batches and shuffles them, and most of the time someone has already built the `Dataset` for you. Next up: **transfer learning** — starting from a model that already knows a lot instead of training from scratch.